# Evaluation Metrics for Ground Truth Documents

This notebook calculates comprehensive evaluation metrics for the two ground truth documents:
- `gpt-4o-mini.txt`
- `groq-llama.txt`

## Metrics Calculated:
1. **ROUGE Scores (ROUGE-1, ROUGE-2, ROUGE-L)**: Bidirectional comparison between the two ground truth documents
2. **Redundancy Score**: Measures repetition and redundancy within each document
3. **Key-Entity Overlap**: Measures overlap of important entities between the two documents
4. **Compression Ratio**: Ratio of each document's length to the source documents length

## Input:
- `ground_truth/gpt-4o-mini.txt`
- `ground_truth/groq-llama.txt`
- Source documents: All `no_footer_*.txt` files from `pre_proc_op/`


In [19]:
# Import required libraries
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# For entity extraction and text processing
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.chunk import ne_chunk
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# ROUGE evaluation
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
    print("Using rouge_score library")
except ImportError:
    ROUGE_AVAILABLE = False
    print("rouge_score not available, installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "rouge-score"])
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

try:
    nltk.data.find('chunkers/maxent_ne_chunker')
except LookupError:
    nltk.download('maxent_ne_chunker', quiet=True)

try:
    nltk.data.find('corpora/words')
except LookupError:
    nltk.download('words', quiet=True)

print("Libraries imported successfully")


Using rouge_score library
Libraries imported successfully


In [20]:
# Configuration - resolve project root (notebook lives in eval/ subfolder)
if (Path.cwd() / "OP_SUMMARIES").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "OP_SUMMARIES").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
EVAL_OUTPUT_DIR = PROJECT_ROOT / "eval"

ground_truth_path = PROJECT_ROOT / "ground_truth"
source_base_path = PROJECT_ROOT / "pre_proc_op"
years = ["2020", "2021", "2022", "2023", "2024"]

# Ground truth files
gpt4o_mini_path = ground_truth_path / "gpt-4o-mini.txt"
groq_llama_path = ground_truth_path / "groq-llama.txt"

# Initialize ROUGE scorer
if ROUGE_AVAILABLE:
    rouge_scorer_instance = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
else:
    rouge_scorer_instance = None

print("Configuration:")
print(f"Ground truth path: {ground_truth_path}")
print(f"Source documents path: {source_base_path}")
print(f"ROUGE library available: {ROUGE_AVAILABLE}")
print(f"\nGround truth files:")
print(f"  GPT-4o-mini: {gpt4o_mini_path}")
print(f"  Groq-Llama: {groq_llama_path}")


Configuration:
Ground truth path: ground_truth
Source documents path: pre_proc_op
ROUGE library available: True

Ground truth files:
  GPT-4o-mini: ground_truth\gpt-4o-mini.txt
  Groq-Llama: ground_truth\groq-llama.txt


In [21]:
# Load ground truth documents
def load_text(file_path):
    """Load text from file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            return content
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return ""

# Load ground truth documents
gpt4o_mini_text = load_text(gpt4o_mini_path)
groq_llama_text = load_text(groq_llama_path)

print("Ground truth documents loaded:")
print(f"  GPT-4o-mini: {len(gpt4o_mini_text)} characters")
print(f"  Groq-Llama: {len(groq_llama_text)} characters")


Ground truth documents loaded:
  GPT-4o-mini: 1459 characters
  Groq-Llama: 1302 characters


In [22]:
# Load all source documents (no_footer_ files)
def load_source_documents():
    """
    Load all no_footer_*.txt files from all years
    """
    all_text = ""
    total_files = 0
    
    for year in years:
        year_path = source_base_path / year
        if not year_path.exists():
            continue
        
        no_footer_files = list(year_path.glob("no_footer_*.txt"))
        
        for file_path in no_footer_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read().strip()
                    if content:
                        all_text += content + " "
                        total_files += 1
            except Exception as e:
                print(f"Error reading {file_path.name}: {e}")
    
    return all_text.strip(), total_files

source_text, num_source_files = load_source_documents()
source_length = len(source_text)

print(f"Source documents loaded:")
print(f"  Total files: {num_source_files}")
print(f"  Total characters: {source_length:,}")


Source documents loaded:
  Total files: 82
  Total characters: 76,488


## 1. ROUGE Scores (ROUGE-1, ROUGE-2, ROUGE-L)

Bidirectional comparison between the two ground truth documents.


In [23]:
# Calculate ROUGE scores (bidirectional)
if ROUGE_AVAILABLE:
    # GPT-4o-mini as reference, Groq-Llama as candidate
    scores_gpt_ref = rouge_scorer_instance.score(gpt4o_mini_text, groq_llama_text)
    
    # Groq-Llama as reference, GPT-4o-mini as candidate
    scores_groq_ref = rouge_scorer_instance.score(groq_llama_text, gpt4o_mini_text)
    
    print("ROUGE SCORES:")
    print("=" * 70)
    print("\n1. GPT-4o-mini as Reference, Groq-Llama as Candidate:")
    print("-" * 70)
    print(f"  ROUGE-1: Precision={scores_gpt_ref['rouge1'].precision:.4f}, Recall={scores_gpt_ref['rouge1'].recall:.4f}, F1={scores_gpt_ref['rouge1'].fmeasure:.4f}")
    print(f"  ROUGE-2: Precision={scores_gpt_ref['rouge2'].precision:.4f}, Recall={scores_gpt_ref['rouge2'].recall:.4f}, F1={scores_gpt_ref['rouge2'].fmeasure:.4f}")
    print(f"  ROUGE-L: Precision={scores_gpt_ref['rougeL'].precision:.4f}, Recall={scores_gpt_ref['rougeL'].recall:.4f}, F1={scores_gpt_ref['rougeL'].fmeasure:.4f}")
    
    print("\n2. Groq-Llama as Reference, GPT-4o-mini as Candidate:")
    print("-" * 70)
    print(f"  ROUGE-1: Precision={scores_groq_ref['rouge1'].precision:.4f}, Recall={scores_groq_ref['rouge1'].recall:.4f}, F1={scores_groq_ref['rouge1'].fmeasure:.4f}")
    print(f"  ROUGE-2: Precision={scores_groq_ref['rouge2'].precision:.4f}, Recall={scores_groq_ref['rouge2'].recall:.4f}, F1={scores_groq_ref['rouge2'].fmeasure:.4f}")
    print(f"  ROUGE-L: Precision={scores_groq_ref['rougeL'].precision:.4f}, Recall={scores_groq_ref['rougeL'].recall:.4f}, F1={scores_groq_ref['rougeL'].fmeasure:.4f}")
    
    # Average scores (symmetric)
    avg_rouge1 = (scores_gpt_ref['rouge1'].fmeasure + scores_groq_ref['rouge1'].fmeasure) / 2
    avg_rouge2 = (scores_gpt_ref['rouge2'].fmeasure + scores_groq_ref['rouge2'].fmeasure) / 2
    avg_rougeL = (scores_gpt_ref['rougeL'].fmeasure + scores_groq_ref['rougeL'].fmeasure) / 2
    
    print("\n3. Average ROUGE Scores (Symmetric):")
    print("-" * 70)
    print(f"  ROUGE-1 F1: {avg_rouge1:.4f}")
    print(f"  ROUGE-2 F1: {avg_rouge2:.4f}")
    print(f"  ROUGE-L F1: {avg_rougeL:.4f}")
else:
    print("ROUGE library not available")


ROUGE SCORES:

1. GPT-4o-mini as Reference, Groq-Llama as Candidate:
----------------------------------------------------------------------
  ROUGE-1: Precision=0.6279, Recall=0.5696, F1=0.5973
  ROUGE-2: Precision=0.2523, Recall=0.2288, F1=0.2400
  ROUGE-L: Precision=0.2930, Recall=0.2658, F1=0.2788

2. Groq-Llama as Reference, GPT-4o-mini as Candidate:
----------------------------------------------------------------------
  ROUGE-1: Precision=0.5696, Recall=0.6279, F1=0.5973
  ROUGE-2: Precision=0.2288, Recall=0.2523, F1=0.2400
  ROUGE-L: Precision=0.2658, Recall=0.2930, F1=0.2788

3. Average ROUGE Scores (Symmetric):
----------------------------------------------------------------------
  ROUGE-1 F1: 0.5973
  ROUGE-2 F1: 0.2400
  ROUGE-L F1: 0.2788


## 2. Compression Ratio

Compression ratio = Document Length / Source Length

Lower ratio = more compressed (better compression)


In [24]:
# Calculate Compression Ratio
def calculate_compression_ratio(document_text, source_text):
    """
    Calculate compression ratio: document_length / source_length
    Also calculate compression percentage: (1 - ratio) * 100
    """
    doc_len = len(document_text)
    src_len = len(source_text)
    
    if src_len == 0:
        return None, None
    
    ratio = doc_len / src_len
    compression_percent = (1 - ratio) * 100
    
    return ratio, compression_percent

# Calculate for both ground truth documents
gpt4o_ratio, gpt4o_compression = calculate_compression_ratio(gpt4o_mini_text, source_text)
groq_ratio, groq_compression = calculate_compression_ratio(groq_llama_text, source_text)

print("COMPRESSION RATIO:")
print("=" * 70)
print("GPT-4o-mini:")
print(f"  Document length: {len(gpt4o_mini_text):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {gpt4o_ratio:.4f}")
print(f"  Compression: {gpt4o_compression:.2f}%")
print(f"\nGroq-Llama:")
print(f"  Document length: {len(groq_llama_text):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {groq_ratio:.4f}")
print(f"  Compression: {groq_compression:.2f}%")


COMPRESSION RATIO:
GPT-4o-mini:
  Document length: 1,459 characters
  Source length: 76,488 characters
  Compression ratio: 0.0191
  Compression: 98.09%

Groq-Llama:
  Document length: 1,302 characters
  Source length: 76,488 characters
  Compression ratio: 0.0170
  Compression: 98.30%


## 3. Key-Entity Overlap

Measures the overlap of important entities between the two ground truth documents.

KE Overlap = (Common Entities / Total Unique Entities in Both Documents) * 100


In [25]:
# Extract key entities from text
def extract_entities(text):
    """
    Extract key entities from text:
    1. Named entities (using NLTK)
    2. Important terms (capitalized words, car numbers, driver names, etc.)
    3. Key technical terms
    """
    entities = set()
    
    # Tokenize and tag
    try:
        tokens = word_tokenize(text)
        tagged = pos_tag(tokens)
        
        # Extract named entities
        tree = ne_chunk(tagged, binary=False)
        for subtree in tree:
            if isinstance(subtree, nltk.Tree):
                entity = ' '.join([word for word, tag in subtree.leaves()])
                entities.add(entity.lower())
    except Exception as e:
        pass  # Silently continue if NER fails
    
    # Extract car numbers (Car 44, Car 63, etc.)
    car_numbers = re.findall(r'car\s+\d+', text, re.IGNORECASE)
    for car in car_numbers:
        entities.add(car.lower())
    
    # Extract driver names (common patterns)
    driver_patterns = [
        r'lewis\s+hamilton',
        r'george\s+russell',
        r'valtteri\s+bottas',
        r'hamilton',
        r'russell',
        r'bottas'
    ]
    for pattern in driver_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            entities.add(match.lower())
    
    # Extract important capitalized terms (likely entities)
    capitalized_terms = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
    for term in capitalized_terms:
        # Filter out common words
        if len(term) > 3 and term.lower() not in ['the', 'and', 'for', 'are', 'was', 'were']:
            entities.add(term.lower())
    
    # Extract technical terms (FIA, Article, etc.)
    technical_terms = re.findall(r'\b(?:fia|article|art\.|regulation|steward|penalty|fine|decision|offence|infringement)\b', text, re.IGNORECASE)
    for term in technical_terms:
        entities.add(term.lower())
    
    return entities

# Calculate Key-Entity Overlap between the two documents
def calculate_ke_overlap_between_docs(doc1_text, doc2_text):
    """
    Calculate Key-Entity Overlap between two documents
    """
    doc1_entities = extract_entities(doc1_text)
    doc2_entities = extract_entities(doc2_text)
    
    # Find common entities
    common_entities = doc1_entities.intersection(doc2_entities)
    
    # Total unique entities in both documents
    all_entities = doc1_entities.union(doc2_entities)
    
    if len(all_entities) == 0:
        return 0.0, doc1_entities, doc2_entities, common_entities, all_entities
    
    # KE Overlap = (Common Entities / Total Unique Entities) * 100
    overlap_score = (len(common_entities) / len(all_entities)) * 100
    
    return overlap_score, doc1_entities, doc2_entities, common_entities, all_entities

# Calculate KE Overlap between the two documents
ke_overlap, gpt4o_entities, groq_entities, common_entities, all_entities = calculate_ke_overlap_between_docs(
    gpt4o_mini_text, groq_llama_text
)

print("KEY-ENTITY OVERLAP (Between Ground Truth Documents):")
print("=" * 70)
print(f"Entities in GPT-4o-mini: {len(gpt4o_entities)}")
print(f"Entities in Groq-Llama: {len(groq_entities)}")
print(f"Common entities: {len(common_entities)}")
print(f"Total unique entities (both documents): {len(all_entities)}")
print(f"KE Overlap: {ke_overlap:.2f}%")
print(f"\nSample common entities: {list(common_entities)[:15]}")


KEY-ENTITY OVERLAP (Between Ground Truth Documents):
Entities in GPT-4o-mini: 14
Entities in Groq-Llama: 12
Common entities: 9
Total unique entities (both documents): 17
KE Overlap: 52.94%

Sample common entities: ['mercedes', 'russell', 'george russell', 'infringement', 'lewis hamilton', 'session', 'hamilton', 'over', 'fia']


## 3b. Key-Entity Overlap with Source Documents

Measures the overlap of important entities between each ground truth document and the source documents.

KE Overlap = (Common Entities / Total Entities in Ground Truth Document) * 100


In [26]:
# Calculate Key-Entity Overlap with source documents (same as eval_metrics.ipynb)
def calculate_ke_overlap(summary_text, source_text):
    """
    Calculate Key-Entity Overlap between summary and source
    KE Overlap = (Common Entities / Total Entities in Summary) * 100
    """
    summary_entities = extract_entities(summary_text)
    source_entities = extract_entities(source_text)
    
    if len(summary_entities) == 0:
        return 0.0, summary_entities, source_entities, set()
    
    # Find common entities
    common_entities = summary_entities.intersection(source_entities)
    
    # KE Overlap = (Common Entities / Total Entities in Summary) * 100
    overlap_score = (len(common_entities) / len(summary_entities)) * 100
    
    return overlap_score, summary_entities, source_entities, common_entities

# Calculate KE Overlap with source for both ground truth documents
gpt4o_ke, gpt4o_sum_ents, gpt4o_src_ents, gpt4o_common = calculate_ke_overlap(gpt4o_mini_text, source_text)
groq_ke, groq_sum_ents, groq_src_ents, groq_common = calculate_ke_overlap(groq_llama_text, source_text)

print("KEY-ENTITY OVERLAP (with Source Documents):")
print("=" * 70)
print("GPT-4o-mini:")
print(f"  Entities in document: {len(gpt4o_sum_ents)}")
print(f"  Entities in source: {len(gpt4o_src_ents)}")
print(f"  Common entities: {len(gpt4o_common)}")
print(f"  KE Overlap: {gpt4o_ke:.2f}%")
print(f"\nGroq-Llama:")
print(f"  Entities in document: {len(groq_sum_ents)}")
print(f"  Entities in source: {len(groq_src_ents)}")
print(f"  Common entities: {len(groq_common)}")
print(f"  KE Overlap: {groq_ke:.2f}%")
print(f"\nSample common entities (GPT-4o-mini): {list(gpt4o_common)[:15]}")
print(f"Sample common entities (Groq-Llama): {list(groq_common)[:15]}")


KEY-ENTITY OVERLAP (with Source Documents):
GPT-4o-mini:
  Entities in document: 14
  Entities in source: 387
  Common entities: 9
  KE Overlap: 64.29%

Groq-Llama:
  Entities in document: 12
  Entities in source: 387
  Common entities: 8
  KE Overlap: 66.67%

Sample common entities (GPT-4o-mini): ['mercedes', 'russell', 'george russell', 'valtteri bottas', 'infringement', 'lewis hamilton', 'bottas', 'hamilton', 'fia']
Sample common entities (Groq-Llama): ['mercedes', 'russell', 'george russell', 'penalty', 'infringement', 'lewis hamilton', 'hamilton', 'fia']


## 4. Redundancy Score

Measures repetition and redundancy within each document using:
- Sentence similarity (cosine similarity between sentences)
- N-gram repetition
- Average sentence similarity


In [27]:
# Calculate Redundancy
def calculate_redundancy(document_text):
    """
    Calculate redundancy metrics:
    1. Average sentence similarity (higher = more redundant)
    2. N-gram repetition rate
    3. Redundancy score (0-100, higher = more redundant)
    """
    if not document_text or len(document_text.strip()) < 10:
        return None, None, None
    
    # Split into sentences
    sentences = sent_tokenize(document_text)
    
    if len(sentences) < 2:
        return 0.0, 0.0, 0.0  # No redundancy if only one sentence
    
    # Calculate sentence similarity using TF-IDF
    vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
    try:
        sentence_vectors = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(sentence_vectors)
        
        # Average similarity (excluding diagonal)
        np.fill_diagonal(similarity_matrix, 0)  # Remove self-similarity
        num_pairs = len(sentences) * (len(sentences) - 1) / 2
        avg_similarity = similarity_matrix.sum() / (2 * num_pairs) if num_pairs > 0 else 0
    except Exception as e:
        print(f"Error calculating similarity: {e}")
        avg_similarity = 0.0
    
    # Calculate n-gram repetition (bigrams and trigrams)
    words = word_tokenize(document_text.lower())
    
    # Bigram repetition
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    bigram_counts = Counter(bigrams)
    repeated_bigrams = sum(1 for count in bigram_counts.values() if count > 1)
    bigram_repetition = (repeated_bigrams / len(bigram_counts)) * 100 if len(bigram_counts) > 0 else 0
    
    # Trigram repetition
    trigrams = [f"{words[i]} {words[i+1]} {words[i+2]}" for i in range(len(words)-2)]
    trigram_counts = Counter(trigrams)
    repeated_trigrams = sum(1 for count in trigram_counts.values() if count > 1)
    trigram_repetition = (repeated_trigrams / len(trigram_counts)) * 100 if len(trigram_counts) > 0 else 0
    
    # Overall redundancy score (weighted combination)
    # Higher score = more redundant
    redundancy_score = (avg_similarity * 50) + (bigram_repetition * 0.3) + (trigram_repetition * 0.2)
    redundancy_score = min(100, redundancy_score)  # Cap at 100
    
    return avg_similarity, bigram_repetition, redundancy_score

# Calculate redundancy for both documents
gpt4o_avg_sim, gpt4o_bigram, gpt4o_redundancy = calculate_redundancy(gpt4o_mini_text)
groq_avg_sim, groq_bigram, groq_redundancy = calculate_redundancy(groq_llama_text)

print("REDUNDANCY SCORE:")
print("=" * 70)
print("GPT-4o-mini:")
print(f"  Average sentence similarity: {gpt4o_avg_sim:.4f}")
print(f"  Bigram repetition rate: {gpt4o_bigram:.2f}%")
print(f"  Redundancy score: {gpt4o_redundancy:.2f} (higher = more redundant)")
print(f"\nGroq-Llama:")
print(f"  Average sentence similarity: {groq_avg_sim:.4f}")
print(f"  Bigram repetition rate: {groq_bigram:.2f}%")
print(f"  Redundancy score: {groq_redundancy:.2f} (higher = more redundant)")


REDUNDANCY SCORE:
GPT-4o-mini:
  Average sentence similarity: 0.0477
  Bigram repetition rate: 13.76%
  Redundancy score: 7.77 (higher = more redundant)

Groq-Llama:
  Average sentence similarity: 0.0854
  Bigram repetition rate: 13.02%
  Redundancy score: 9.42 (higher = more redundant)


## Summary Table

Comprehensive comparison of all metrics for both ground truth documents.


In [28]:
# Create summary table
if ROUGE_AVAILABLE:
    # Explanation: 
    # - For GPT-4o-mini row: Shows ROUGE scores when Groq-Llama is the reference (i.e., how well GPT-4o-mini matches Groq-Llama)
    # - For Groq-Llama row: Shows ROUGE scores when GPT-4o-mini is the reference (i.e., how well Groq-Llama matches GPT-4o-mini)
    
    summary_data = {
        'Document': ['GPT-4o-mini', 'Groq-Llama'],
        'Length (chars)': [len(gpt4o_mini_text), len(groq_llama_text)],
        'Compression Ratio': [f"{gpt4o_ratio:.4f}", f"{groq_ratio:.4f}"],
        'Compression %': [f"{gpt4o_compression:.2f}%", f"{groq_compression:.2f}%"],
        'KE Overlap (vs Source)': [f"{gpt4o_ke:.2f}%", f"{groq_ke:.2f}%"],
        'Redundancy Score': [f"{gpt4o_redundancy:.2f}", f"{groq_redundancy:.2f}"],
        'ROUGE-1 F1': [
            f"{scores_groq_ref['rouge1'].fmeasure:.4f}",
            f"{scores_gpt_ref['rouge1'].fmeasure:.4f}"
        ],
        'ROUGE-2 F1': [
            f"{scores_groq_ref['rouge2'].fmeasure:.4f}",
            f"{scores_gpt_ref['rouge2'].fmeasure:.4f}"
        ],
        'ROUGE-L F1': [
            f"{scores_groq_ref['rougeL'].fmeasure:.4f}",
            f"{scores_gpt_ref['rougeL'].fmeasure:.4f}"
        ]
    }
    
    df_summary = pd.DataFrame(summary_data)
    print("SUMMARY TABLE:")
    print("=" * 120)
    print("Note: ROUGE scores show how well each document matches the OTHER document as reference")
    print("      - GPT-4o-mini scores: evaluated against Groq-Llama as reference")
    print("      - Groq-Llama scores: evaluated against GPT-4o-mini as reference")
    print("-" * 120)
    print(df_summary.to_string(index=False))
    
    # Additional metrics
    print("\n\nADDITIONAL METRICS:")
    print("=" * 70)
    print(f"Key-Entity Overlap (between ground truth documents): {ke_overlap:.2f}%")
    print(f"Key-Entity Overlap - GPT-4o-mini (vs source): {gpt4o_ke:.2f}%")
    print(f"Key-Entity Overlap - Groq-Llama (vs source): {groq_ke:.2f}%")
    print(f"Average ROUGE-1 F1 (symmetric): {avg_rouge1:.4f}")
    print(f"Average ROUGE-2 F1 (symmetric): {avg_rouge2:.4f}")
    print(f"Average ROUGE-L F1 (symmetric): {avg_rougeL:.4f}")
    
    # Save results to CSV
    df_summary.to_csv(EVAL_OUTPUT_DIR / 'ground_truth_evaluation_results.csv', index=False)
    print(f"\n\nResults saved to '{EVAL_OUTPUT_DIR / 'ground_truth_evaluation_results.csv'}")
else:
    print("ROUGE scores not available for summary table")


SUMMARY TABLE:
Note: ROUGE scores show how well each document matches the OTHER document as reference
      - GPT-4o-mini scores: evaluated against Groq-Llama as reference
      - Groq-Llama scores: evaluated against GPT-4o-mini as reference
------------------------------------------------------------------------------------------------------------------------
   Document  Length (chars) Compression Ratio Compression % KE Overlap (vs Source) Redundancy Score ROUGE-1 F1 ROUGE-2 F1 ROUGE-L F1
GPT-4o-mini            1459            0.0191        98.09%                 64.29%             7.77     0.5973     0.2400     0.2788
 Groq-Llama            1302            0.0170        98.30%                 66.67%             9.42     0.5973     0.2400     0.2788


ADDITIONAL METRICS:
Key-Entity Overlap (between ground truth documents): 52.94%
Key-Entity Overlap - GPT-4o-mini (vs source): 64.29%
Key-Entity Overlap - Groq-Llama (vs source): 66.67%
Average ROUGE-1 F1 (symmetric): 0.5973
Average ROUG